# Day 052 — Exercise 4: Path & Query Parameters

**What you'll build:** `create_template_app()` — a prompt-template service. `GET /templates` lists names; `GET /render/{name}?topic=...` renders a prompt from a **path** parameter and a **query** parameter; an unknown name raises `HTTPException(404)`.

**Why it matters:** Not every input comes in a JSON body. Path parameters (`/render/{name}`) identify a resource; query parameters (`?topic=...`) tune the request and can have defaults. FastAPI reads both straight from the function signature. And returning the right status code — 404 for 'not found' — is what makes an API predictable to its callers.

## Provided: Setup + Models

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama


class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str

## Your Implementation

In [ ]:
PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def create_template_app() -> FastAPI:
    """
    GET /templates              -> {'templates': [names]}
    GET /render/{name}?topic=.. -> {'name': name, 'prompt': rendered}
    Unknown name                -> HTTPException(404)
    """
    app = FastAPI()

    @app.get('/templates')
    def list_templates():
        # TODO: return {'templates': list(PROMPT_TEMPLATES.keys())}
        pass

    # TODO: define GET /render/{name} with a query param topic (default 'AI'):
    # @app.get('/render/{name}')
    # def render(name: str, topic: str = 'AI'):
    #     if name not in PROMPT_TEMPLATES:
    #         raise HTTPException(status_code=404, detail=f'template {name!r} not found')
    #     return {'name': name, 'prompt': PROMPT_TEMPLATES[name].format(topic=topic)}

    return app

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    try:
        client = TestClient(create_template_app())
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 1: GET /templates lists the template names
    try:
        r = client.get('/templates')
        assert r.status_code == 200, f'expected 200, got {r.status_code}'
        assert 'summary' in r.json()['templates'], f"missing 'summary': {r.json()}"
        passed += 1; print('✅ Check 1: GET /templates lists names')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: path param + query param render a prompt with the topic
    try:
        r = client.get('/render/summary', params={'topic': 'Python'})
        assert r.status_code == 200, f'expected 200, got {r.status_code}'
        assert 'Python' in r.json()['prompt'], f"topic not rendered: {r.json()}"
        passed += 1; print('✅ Check 2: /render/{name}?topic= renders the prompt')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: query param has a default of 'AI'
    try:
        r = client.get('/render/explain')
        assert r.status_code == 200
        assert 'AI' in r.json()['prompt'], f"default topic not applied: {r.json()}"
        passed += 1; print('✅ Check 3: topic defaults to AI')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: unknown template name -> 404
    try:
        r = client.get('/render/does-not-exist')
        assert r.status_code == 404, f'expected 404, got {r.status_code}'
        passed += 1; print('✅ Check 4: unknown template -> 404')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: the path param is reflected back in the response
    try:
        r = client.get('/render/critique')
        assert r.json().get('name') == 'critique', f"name not reflected: {r.json()}"
        passed += 1; print('✅ Check 5: path param reflected in response')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def create_template_app() -> FastAPI:
    """A prompt-template service using PATH and QUERY parameters.

    GET /templates                    -> list of template names
    GET /render/{name}?topic=...      -> the rendered prompt string
    Unknown template name             -> HTTPException(404)
    """
    app = FastAPI()

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.get('/render/{name}')
    def render(name: str, topic: str = 'AI'):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        return {'name': name, 'prompt': PROMPT_TEMPLATES[name].format(topic=topic)}

    return app
```

**Why this works:** In `/render/{name}`, `name` appears in the path, so FastAPI binds it as a path parameter. `topic: str = 'AI'` is *not* in the path, so FastAPI treats it as a query parameter with a default — `/render/summary` and `/render/summary?topic=Python` both work. When the name isn't a known template, `raise HTTPException(404)` returns the correct 'not found' status instead of a generic error. Clean status codes are the difference between an API and a black box.
</details>